# Pipeline v20.1 -- PURE QWEN inference/eval
**EXACT 2026 -- data v5 strict-clean**

Final answer path is **pure Qwen3-8B LoRA v20** only:
- no Z3
- no hybrid
- no anti-overclaim
- no LGBM
- BoN self-consistency: N=5, temp=0.5, top_p=0.95


In [ ]:

# ==================================================================
# STAGE 0 -- vLLM compatibility installer (Kaggle T4)
# Must run before any torch/vllm imports in a clean kernel.
# ==================================================================
import os, sys, subprocess, importlib

os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

def _pip(*args):
    return subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "--break-system-packages"] + list(args),
        capture_output=True, text=True
    )

def _clear_vllm():
    for mod in list(sys.modules.keys()):
        if any(x in mod for x in ("vllm", "transformers", "torch._dynamo", "torch._inductor")):
            del sys.modules[mod]

def _try_import():
    try:
        from vllm import LLM, SamplingParams
        return True, ""
    except Exception as e:
        return False, str(e).split("\n")[0][:180]

print("="*80)
print("STAGE 0 vLLM compatibility installer")
print("="*80)

ok, msg = _try_import()
if ok:
    print("Pre-installed vLLM: OK", flush=True)
    _vllm_ok = True
else:
    print(f"Pre-installed vLLM: FAIL ({msg})", flush=True)
    PAIRS = [
        ("4.48.0", ""),             # worked in previous v17/v18 runs: latest vLLM
        ("4.57.6", "0.22.1"),
        ("4.48.0", "0.22.1"),
        ("4.46.3", "0.9.1"),
        ("4.46.3", "0.8.5.post1"),
        ("4.44.2", "0.8.5.post1"),
        ("4.44.2", "0.7.3"),
        ("4.46.3", "0.6.6.post1"),
    ]
    _vllm_ok = False
    for tv, vv in PAIRS:
        vllm_pkg = f"vllm=={vv}" if vv else "vllm"
        print(f"  Trying transformers=={tv} + {vllm_pkg}...", end=" ", flush=True)
        _pip("protobuf==5.29.5")
        _pip(f"transformers=={tv}")
        _pip(vllm_pkg)
        _clear_vllm()
        ok, msg = _try_import()
        if ok:
            print("OK", flush=True)
            _vllm_ok = True
            break
        else:
            print(f"FAIL ({msg})", flush=True)
    if not _vllm_ok:
        raise RuntimeError("No vLLM+transformers pair worked. Restart session and retry, or use transformers fallback.")

# Final imports
import json, re, csv, time, random, glob, zipfile, os, sys
from pathlib import Path
from collections import Counter, defaultdict

import torch
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest
from transformers import AutoTokenizer
import vllm as _vllm_mod
import transformers as _tfm

N_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0

print("\n" + "="*80)
print("IMPORT SUMMARY")
print("="*80)
print(f"PyTorch      : {torch.__version__}")
print(f"CUDA OK      : {torch.cuda.is_available()}")
print(f"N_GPUS       : {N_GPUS}")
for i in range(N_GPUS):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}        : {props.name} ({props.total_memory/1024**3:.1f} GB)")
print(f"vLLM version : {_vllm_mod.__version__}")
print(f"Transformers : {_tfm.__version__}")
print("All imports OK", flush=True)


In [ ]:

# ==================================================================
# BOOT CHECK
# ==================================================================
import os, sys, glob, torch
print("="*70)
print("BOOT CHECK")
print("="*70)
print("Python:", sys.version.split()[0])
print("Torch :", torch.__version__, "CUDA available=", torch.cuda.is_available())
print("N_GPUS:", N_GPUS)
print("/kaggle/input entries:")
for p in sorted(glob.glob("/kaggle/input/*"))[:30]:
    print(" -", p)
print("="*70)


In [ ]:

# ==================================================================
# LOCATE + UNZIP LoRA v20
# ==================================================================
import os, glob, zipfile

LORA_ZIP_GLOB = "/kaggle/input/**/qwen3_cot_lora_v20_v5.zip"
UNZIP_DEST = "/kaggle/working"
zip_hits = sorted(glob.glob(LORA_ZIP_GLOB, recursive=True))

print("="*70)
print("LOCATE LoRA v20")
print("="*70)
print("zip hits:", zip_hits[:5])

resolved = None
if zip_hits:
    zpath = zip_hits[0]
    print("Found zip:", zpath)
    with zipfile.ZipFile(zpath) as zf:
        zf.extractall(UNZIP_DEST)
    cand = os.path.join(UNZIP_DEST, "qwen3_cot_lora_v20_v5")
    if os.path.exists(os.path.join(cand, "adapter_config.json")):
        resolved = cand
    else:
        for root, _, files in os.walk(UNZIP_DEST):
            if "adapter_config.json" in files and "qwen3_cot_lora_v20" in root:
                resolved = root
                break
else:
    for root, _, files in os.walk("/kaggle/input"):
        if "adapter_config.json" in files and "qwen3_cot_lora_v20" in root:
            resolved = root
            break

assert resolved is not None, "Cannot find qwen3_cot_lora_v20_v5.zip or unzipped adapter. Add v20 finetune output as input."
assert os.path.exists(os.path.join(resolved, "adapter_config.json")), resolved
assert os.path.exists(os.path.join(resolved, "adapter_model.safetensors")), resolved

COT_LORA_PATH = resolved
print("[OK] COT_LORA_PATH =", COT_LORA_PATH)
print("[OK] adapter_model.safetensors MB =", os.path.getsize(os.path.join(resolved, "adapter_model.safetensors"))/1e6)


In [ ]:

# ==================================================================
# CONFIG + PATH RESOLVER
# ==================================================================
import os, glob, random, json, re, csv, time
from collections import Counter, defaultdict

def _resolve(cands, label="path"):
    expanded = []
    for p in cands:
        hits = sorted(glob.glob(p, recursive=True)) if any(ch in p for ch in "*?[") else [p]
        expanded.extend(hits)
    for p in expanded:
        if os.path.exists(p):
            print(f"resolved {label}: {p}")
            return p
    print(f"WARNING {label}: none of candidates exist; using first: {cands[0]}")
    return cands[0]

DATASET_PATH = _resolve([
    "/kaggle/input/**/Logic_Based_Educational_Queries_v5_repair_tier1_dedup_normfol_drop_logicfallacy.json",
    "/kaggle/input/test-pipeline/Logic_Based_Educational_Queries_v5_repair_tier1_dedup_normfol_drop_logicfallacy.json",
    "/kaggle/input/datasets/nguyenminhtric/test-pipeline/Logic_Based_Educational_Queries_v5_repair_tier1_dedup_normfol_drop_logicfallacy.json",
], "DATASET")

TEST_PATH = _resolve([
    "/kaggle/input/**/generated_v4style_300.json",
    "/kaggle/input/test-pipeline/generated_v4style_300.json",
    "/kaggle/input/datasets/nguyenminhtric/test-pipeline/generated_v4style_300.json",
], "TEST")

QWEN_MODEL_ID = "/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1"

SEED = 42
TRAIN_RATIO = 0.80
QWEN_BON_N = 5
QWEN_BON_TEMP = 0.5
QWEN_TOP_P = 0.95
COT_MAX_TOKENS = 768
TENSOR_PARALLEL = min(max(N_GPUS, 1), 2)
MAX_MODEL_LEN = 8192
GPU_MEMORY_UTIL = 0.85
OUTPUT_JSON = "/kaggle/working/pipeline_results_20_1_pureqwen_v5.json"
SUBMISSION_CSV = "/kaggle/working/submission_v20_pureqwen.csv"
SUBMISSION_JSON = "/kaggle/working/submission_v20_pureqwen.json"

V20_COT_SYSTEM = (
    "You are a careful logic tutor. Given a list of premises and a question, "
    "reason step-by-step by referencing specific premises (e.g. 'From premise 3...'). "
    "Then state your conclusion on a final line in the exact format: 'Final Answer: X' "
    "where X is one of: Yes, No, Unknown, A, B, C, or D."
)

print("="*70)
print("PURE QWEN CONFIG")
print("="*70)
print("DATASET_PATH:", DATASET_PATH)
print("TEST_PATH   :", TEST_PATH)
print("COT_LORA_PATH:", COT_LORA_PATH)
print("QWEN_BON_N:", QWEN_BON_N, "temp:", QWEN_BON_TEMP, "top_p:", QWEN_TOP_P)
print("TENSOR_PARALLEL:", TENSOR_PARALLEL)
print("NO Z3 / NO HYBRID / NO LGBM")


In [ ]:

# ==================================================================
# LOAD vLLM + TOKENIZER
# ==================================================================
print("="*70)
print("Loading vLLM engine...")
print("="*70)
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_ID, trust_remote_code=True)

llm = LLM(
    model=QWEN_MODEL_ID,
    tokenizer=QWEN_MODEL_ID,
    trust_remote_code=True,
    dtype="float16",
    tensor_parallel_size=TENSOR_PARALLEL,
    max_model_len=MAX_MODEL_LEN,
    gpu_memory_utilization=GPU_MEMORY_UTIL,
    enable_lora=True,
    max_lora_rank=16,
    enforce_eager=True,
)

LORA_REQ = LoRARequest("v20_cot_lora", 1, COT_LORA_PATH)
print(f"vLLM loaded in {(time.time()-t0):.1f}s")
print("LoRA enabled:", COT_LORA_PATH)


In [ ]:

# ==================================================================
# DATA LOADING + PROMPT BUILDING
# ==================================================================
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

raw = load_json(DATASET_PATH)
print(f"Loaded labeled data: {len(raw)} records")

rng = random.Random(SEED)
ids = list(range(len(raw)))
rng.shuffle(ids)
n_train = int(len(raw) * TRAIN_RATIO)
train_ids = set(ids[:n_train])
val_ids = set(ids[n_train:])

def iter_questions(samples, split_name, keep_gold=True):
    rows = []
    for sid, s in enumerate(samples):
        nls = s.get("premises-NL", [])
        qs = s.get("questions", [])
        ans = s.get("answers", [])
        exps = s.get("explanation", [])
        for q_idx, q in enumerate(qs):
            gold = str(ans[q_idx]).strip() if keep_gold and q_idx < len(ans) else None
            exp  = str(exps[q_idx]) if q_idx < len(exps) else ""
            rows.append({
                "sample_id": sid, "q_idx": q_idx, "split": split_name,
                "premises_nl": nls, "question": q, "gold": gold, "explanation": exp
            })
    return rows

train_rows = iter_questions([raw[i] for i in range(len(raw)) if i in train_ids], "train", True)
val_rows   = iter_questions([raw[i] for i in range(len(raw)) if i in val_ids], "val", True)
all_rows   = iter_questions(raw, "all", True)

print(f"Train questions: {len(train_rows)}")
print(f"Val questions  : {len(val_rows)}")
print(f"All questions  : {len(all_rows)}")
print("Val label distribution:", Counter(r["gold"] for r in val_rows))

def build_user_msg(premises_nl, question):
    lines = ["### Premises"]
    for i, p in enumerate(premises_nl):
        lines.append(f"P{i+1}. {p}")
    return "\n".join(lines) + f"\n\n### Question\n{question}"

def to_chat_prompt(row):
    messages = [
        {"role": "system", "content": V20_COT_SYSTEM},
        {"role": "user", "content": build_user_msg(row["premises_nl"], row["question"])},
    ]
    try:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
        )
    except TypeError:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


In [ ]:

# ==================================================================
# PURE QWEN BoN GENERATION + ANSWER EXTRACTION
# ==================================================================
VALID = {"YES":"Yes", "NO":"No", "UNKNOWN":"Unknown", "A":"A", "B":"B", "C":"C", "D":"D"}

def normalize_answer(x):
    if x is None:
        return "Unknown"
    s = str(x).strip()
    u = s.upper()
    if u in VALID:
        return VALID[u]
    if u in ("TRUE", "T"):
        return "Yes"
    if u in ("FALSE", "F"):
        return "No"
    if "UNKNOWN" in u or "CANNOT BE DETERMINED" in u or "NOT ENOUGH" in u:
        return "Unknown"
    return "Unknown"

def extract_final_answer(text):
    if not text:
        return "Unknown"
    # Prefer explicit final answer marker.
    pats = [
        r"Final\s*Answer\s*:\s*(Yes|No|Unknown|A|B|C|D)\b",
        r"Final\s*answer\s*is\s*(Yes|No|Unknown|A|B|C|D)\b",
        r"Answer\s*:\s*(Yes|No|Unknown|A|B|C|D)\b",
    ]
    for pat in pats:
        m = re.search(pat, text, flags=re.I)
        if m:
            return normalize_answer(m.group(1))
    # Fallback: last occurrence of a valid standalone label near the end.
    tail = text[-300:]
    hits = re.findall(r"\b(Yes|No|Unknown|A|B|C|D)\b", tail, flags=re.I)
    return normalize_answer(hits[-1]) if hits else "Unknown"

def majority_vote(ans_list):
    norm = [normalize_answer(a) for a in ans_list]
    cnt = Counter(norm)
    # deterministic tie-break: first answer among tied labels
    best_n = max(cnt.values()) if cnt else 0
    tied = {k for k,v in cnt.items() if v == best_n}
    for a in norm:
        if a in tied:
            return a, cnt
    return "Unknown", cnt

def batch_predict(rows, batch_size=128):
    prompts = [to_chat_prompt(r) for r in rows]
    params = SamplingParams(
        n=QWEN_BON_N,
        temperature=QWEN_BON_TEMP,
        top_p=QWEN_TOP_P,
        max_tokens=COT_MAX_TOKENS,
        repetition_penalty=1.05,
    )
    results = []
    t0 = time.time()
    for start in range(0, len(prompts), batch_size):
        end = min(start + batch_size, len(prompts))
        outs = llm.generate(prompts[start:end], params, lora_request=LORA_REQ)
        for row, out in zip(rows[start:end], outs):
            texts = [o.text for o in out.outputs]
            ans_list = [extract_final_answer(t) for t in texts]
            pred, votes = majority_vote(ans_list)
            rec = dict(row)
            rec.update({
                "qwen_answer": pred,
                "qwen_votes": dict(votes),
                "qwen_sc_conf": max(votes.values()) / max(1, sum(votes.values())),
                "qwen_raw_0": texts[0] if texts else "",
            })
            results.append(rec)
        print(f"[{end}/{len(prompts)}] elapsed={(time.time()-t0)/60:.1f} min", flush=True)
    return results


In [ ]:

# ==================================================================
# RUN EVAL ON TRAIN / VAL / FULL
# ==================================================================
train_pred = batch_predict(train_rows)
val_pred   = batch_predict(val_rows)
# Avoid re-running full separately: combine train+val predictions.
all_pred = train_pred + val_pred

def _U(x): return str(x).strip().upper()

def compute_metrics(rows):
    preds = [_U(r["qwen_answer"]) for r in rows]
    golds = [_U(r["gold"]) for r in rows]
    labels = sorted(set(golds))
    per = {}
    for lab in labels:
        tp = sum(1 for p,g in zip(preds,golds) if p==lab and g==lab)
        fp = sum(1 for p,g in zip(preds,golds) if p==lab and g!=lab)
        fn = sum(1 for p,g in zip(preds,golds) if p!=lab and g==lab)
        prec = tp/(tp+fp) if tp+fp else 0.0
        rec = tp/(tp+fn) if tp+fn else 0.0
        f1 = 2*prec*rec/(prec+rec) if prec+rec else 0.0
        per[lab] = {"precision":prec, "recall":rec, "f1":f1, "n":sum(1 for g in golds if g==lab)}
    acc = sum(1 for p,g in zip(preds,golds) if p==g) / len(rows) if rows else 0.0
    macro_f1 = sum(v["f1"] for v in per.values()) / len(per) if per else 0.0
    weighted_f1 = sum(v["f1"]*v["n"] for v in per.values()) / len(rows) if rows else 0.0
    mc_labels = {"A","B","C","D"}
    mc_rows = [r for r in rows if _U(r["gold"]) in mc_labels]
    ynu_rows = [r for r in rows if _U(r["gold"]) in {"YES","NO","UNKNOWN"}]
    return {
        "acc": acc, "macro_f1": macro_f1, "weighted_f1": weighted_f1, "per_class": per,
        "mc_acc": (sum(_U(r["qwen_answer"])==_U(r["gold"]) for r in mc_rows)/len(mc_rows) if mc_rows else 0.0),
        "ynu_acc": (sum(_U(r["qwen_answer"])==_U(r["gold"]) for r in ynu_rows)/len(ynu_rows) if ynu_rows else 0.0),
        "n": len(rows),
    }

m_train = compute_metrics(train_pred)
m_val   = compute_metrics(val_pred)
m_full  = compute_metrics(all_pred)

def f1(m, lab):
    return m["per_class"].get(lab, {}).get("f1", 0.0)

print("\n" + "="*100)
print("v20.1 PURE QWEN -- metrics")
print("="*100)
print(f"{'Split':8s} | {'acc':>7s} {'mF1':>7s} {'wF1':>7s} {'Yes-F1':>7s} {'No-F1':>7s} {'Unk-F1':>7s} {'MC-acc':>7s}")
print("-"*100)
for name,m in [("TRAIN",m_train),("VAL",m_val),("FULL",m_full)]:
    print(f"{name:8s} | {m['acc']*100:6.1f}% {m['macro_f1']:7.3f} {m['weighted_f1']:7.3f} "
          f"{f1(m,'YES'):7.3f} {f1(m,'NO'):7.3f} {f1(m,'UNKNOWN'):7.3f} {m['mc_acc']*100:6.1f}%")
print("="*100)

print("\nPer-class VAL:")
for lab, d in m_val["per_class"].items():
    print(f"  {lab:>8s}: f1={d['f1']:.3f}  prec={d['precision']:.3f}  rec={d['recall']:.3f}  n={d['n']}")


In [ ]:

# ==================================================================
# SAVE RESULTS
# ==================================================================
payload = {
    "meta": {
        "version": "v20_1_pure_qwen_v5",
        "model": "Qwen3-8B + LoRA v20",
        "answer_path": "pure_qwen_only",
        "no_z3_no_hybrid_no_lgbm": True,
        "seed": SEED,
        "train_ratio": TRAIN_RATIO,
        "qwen_bon_n": QWEN_BON_N,
        "temperature": QWEN_BON_TEMP,
        "top_p": QWEN_TOP_P,
        "cot_lora_path": COT_LORA_PATH,
        "dataset_path": DATASET_PATH,
    },
    "metrics": {"train": m_train, "val": m_val, "full": m_full},
    "val_predictions": val_pred,
    "train_predictions_sample": train_pred[:20],
}
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("Saved:", OUTPUT_JSON)

# Decision gate
val = m_val
yes_f1 = f1(val, "YES")
no_f1 = f1(val, "NO")
unk_f1 = f1(val, "UNKNOWN")
macro = val["macro_f1"]
print("\n" + "="*70)
print("V20 DECISION GATE")
print("="*70)
print(f"macro-F1   = {macro:.3f}  target >= 0.620")
print(f"No-F1      = {no_f1:.3f}  target >= 0.600")
print(f"Unknown-F1 = {unk_f1:.3f}  target >= 0.550")
print(f"Yes-F1     = {yes_f1:.3f}  target >= 0.750")
if macro >= 0.62 and no_f1 >= 0.60 and unk_f1 >= 0.55 and yes_f1 >= 0.75:
    print("PASS: v20 is a strong final candidate.")
elif macro < 0.538:
    print("FAIL: below v18 macro-F1. Fallback: v20b No 4x, Unknown 1x.")
elif yes_f1 < 0.70:
    print("WARNING: Yes-F1 low. Fallback: reduce No oversample to 2x.")
elif unk_f1 < 0.50:
    print("WARNING: Unknown-F1 low. Fallback: No 3x, Unknown 1.5x.")
else:
    print("MIXED: compare against v17 aggressive and paper defensibility.")
print("="*70)


In [ ]:

# ==================================================================
# OPTIONAL: PREDICT OFFICIAL-LIKE TEST IF PRESENT
# ==================================================================
if os.path.exists(TEST_PATH):
    try:
        test_raw = load_json(TEST_PATH)
        test_rows = iter_questions(test_raw, "test", keep_gold=False)
        print(f"Loaded test: {len(test_raw)} records / {len(test_rows)} questions")
        test_pred = batch_predict(test_rows)
        # CSV: id,answer. Use sample_id_qidx as stable id unless official format differs.
        with open(SUBMISSION_CSV, "w", encoding="utf-8", newline="") as f:
            w = csv.writer(f)
            w.writerow(["id", "answer"])
            for r in test_pred:
                w.writerow([f"{r['sample_id']}_{r['q_idx']}", r["qwen_answer"]])
        with open(SUBMISSION_JSON, "w", encoding="utf-8") as f:
            json.dump(test_pred, f, ensure_ascii=False, indent=2)
        print("Saved:", SUBMISSION_CSV)
        print("Saved:", SUBMISSION_JSON)
    except Exception as e:
        print("Test prediction skipped due to error:", repr(e))
else:
    print("TEST_PATH not found, skip official-like prediction.")
